# Stage 2, step 1: candidate briefings for preference pairs

The stage 1 student writes **two** briefings for every training packet, at a higher temperature than the product uses, so the two differ. Back in the repo, `training/rank_pairs.py` decides which of each pair is better (hard rules first, then a judge) and `04_dpo_colab.ipynb` trains on the ranked pairs.

Output: `MyDrive/news-briefer/candidates.jsonl`, one line per packet with both candidates.

**Runtime → Change runtime type → L4 GPU** (or T4), then **Runtime → Run all**. About 20 minutes on an L4 for 415 packets x 2.


In [ ]:
import subprocess, sys, importlib

USE_UNSLOTH = True  # the fallback cell sets this to False

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *args]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("pip failed -- run the fallback cell below instead")

# unsloth pins its own tested versions of trl / peft / transformers / accelerate /
# bitsandbytes / datasets, so we pin unsloth to a release series and let it choose.
pip("--upgrade", "unsloth==2025.9.*", "unsloth_zoo==2025.9.*")
pip("matplotlib")

print()
print("installed versions:")
for name in ["unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes", "torch"]:
    try:
        mod = importlib.import_module(name)
        print(f"  {name:<14} {getattr(mod, '__version__', '?')}")
    except Exception as exc:  # noqa: BLE001
        print(f"  {name:<14} NOT IMPORTABLE ({exc})")

## 2. The training packets and your Drive

In [ ]:
import json, os, urllib.request
from google.colab import drive

drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/news-briefer"
ADAPTER_DIR = os.path.join(WORK_DIR, "news-briefer-lora")   # the stage 1 adapter
assert os.path.isdir(ADAPTER_DIR), f"adapter not found at {ADAPTER_DIR}"

TRAIN_FILE = "sft_train.jsonl"
if not os.path.exists(TRAIN_FILE):
    urllib.request.urlretrieve(
        "https://github.com/sanjivp2703/adaptive-news-briefing-agent/releases/download/training-data-v7/" + TRAIN_FILE,
        TRAIN_FILE,
    )
with open(TRAIN_FILE, encoding="utf-8") as fh:
    train_rows = [json.loads(line) for line in fh if line.strip()]
print(len(train_rows), "training packets")


## 3. Generate two candidates per packet

Temperature 0.8 (the product uses 0.3): enough that the two candidates differ in real ways, such as one defining a term and the other not, while both stay coherent. Interrupted runs resume: packets already in the output file are skipped.

In [ ]:
import time, torch
from unsloth import FastLanguageModel

CANDIDATES_PER_PACKET = 2
TEMPERATURE = 0.8
MAX_NEW = 1200
OUT = os.path.join(WORK_DIR, "candidates.jsonl")

done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as fh:
        done = {json.loads(l)["id"] for l in fh if l.strip()}
print(len(done), "packets already generated; resuming")

model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=8192, dtype=None, load_in_4bit=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_inference(model)

t_start = time.time()
with open(OUT, "a", encoding="utf-8") as out:
    for i, row in enumerate(train_rows, 1):
        rid = row["meta"]["id"]
        if rid in done:
            continue
        ids = tokenizer.apply_chat_template(row["messages"][:2], add_generation_prompt=True, return_tensors="pt").to(model.device)
        candidates = []
        for _ in range(CANDIDATES_PER_PACKET):
            with torch.no_grad():
                gen = model.generate(input_ids=ids, max_new_tokens=MAX_NEW, temperature=TEMPERATURE, do_sample=True,
                                     top_p=0.95, pad_token_id=tokenizer.pad_token_id)
            candidates.append(tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True))
        out.write(json.dumps({"id": rid, "profile": row["meta"].get("profile"), "candidates": candidates}) + "\n")
        out.flush()
        if i % 10 == 0:
            rate = (time.time() - t_start) / max(1, i - len(done))
            print(f"  {i}/{len(train_rows)}  {rate:4.1f}s per packet  ~{rate * (len(train_rows) - i) / 60:.0f} min left")
print("wrote", OUT)


## 4. Done

`candidates.jsonl` is in `MyDrive/news-briefer/`. Download it to your Mac; the next step runs there:

    PYTHONPATH=src:. .venv/bin/python training/rank_pairs.py ~/Downloads/candidates.jsonl


In [ ]:
from google.colab import files
print(os.path.getsize(OUT) // 1024, "KB")
files.download(OUT)
